In [1]:
import torch
import torch.nn as nn
import torch.nn.functional as F

# ============================================================
# MIXTURE OF EXPERTS LAYER (SIMPLIFIED DEEPSEEK V3 STYLE)
# Key insight: N experts but only K active per token
# ============================================================

class SwiGLUExpert(nn.Module):
    """Single expert FFN with SwiGLU activation (used by Llama, DeepSeek)."""
    def __init__(self, dim, hidden_dim):
        super().__init__()
        self.w1 = nn.Linear(dim, hidden_dim, bias=False)  # Gate projection
        self.w2 = nn.Linear(hidden_dim, dim, bias=False)  # Down projection
        self.w3 = nn.Linear(dim, hidden_dim, bias=False)  # Up projection

    def forward(self, x):
        # SwiGLU: combines gating with feedforward
        return self.w2(F.silu(self.w1(x)) * self.w3(x))

class MoELayer(nn.Module):
    """
    Mixture of Experts layer with top-k routing.

    Architecture (DeepSeek V3 simplified):
 - 1 shared expert (always active)
 - N routed experts (top-K selected per token)
 - Router: simple linear layer -> softmax -> top-K
    """
    def __init__(self, dim, hidden_dim, num_experts=8, top_k=2):
        super().__init__()
        self.num_experts = num_experts
        self.top_k = top_k

        # Router: produces a score for each expert
        self.router = nn.Linear(dim, num_experts, bias=False)

        # Shared expert (always active, like DeepSeek V3)
        self.shared_expert = SwiGLUExpert(dim, hidden_dim)

        # Routed experts (only top-K active per token)
        self.experts = nn.ModuleList([
            SwiGLUExpert(dim, hidden_dim) for _ in range(num_experts)
        ])

    def forward(self, x, return_aux_loss=True):
        batch, seq_len, dim = x.shape
        x_flat = x.view(-1, dim)  # [B*S, D]

        # Step 1: Router scores
        router_logits = self.router(x_flat)  # [B*S, N]
        router_probs = F.softmax(router_logits, dim=-1)

        # Step 2: Top-K selection
        top_k_probs, top_k_idx = torch.topk(
            router_probs, self.top_k, dim=-1
        )
        # Normalize selected weights to sum to 1
        top_k_weights = top_k_probs / top_k_probs.sum(dim=-1, keepdim=True)

        # Step 3: Shared expert (always runs)
        shared_out = self.shared_expert(x_flat)

        # Step 4: Routed experts (only top-K run)
        routed_out = torch.zeros_like(x_flat)
        for i, expert in enumerate(self.experts):
            # Find tokens routed to this expert
            mask = (top_k_idx == i).any(dim=-1)
            if mask.any():
                expert_out = expert(x_flat[mask])
                # Weight by gate probability
                expert_weight = top_k_weights[mask]
                idx_match = (top_k_idx[mask] == i).float()
                weight = (expert_weight * idx_match).sum(dim=-1, keepdim=True)
                routed_out[mask] += weight * expert_out

        # Combine shared + routed
        output = shared_out + routed_out

        # Step 5: Auxiliary load balancing loss
        aux_loss = None
        if return_aux_loss:
            # f_i = fraction of tokens routed to each expert
            expert_counts = torch.zeros(self.num_experts, device=x.device)
            for i in range(self.num_experts):
                expert_counts[i] = (top_k_idx == i).float().sum()
            f_i = expert_counts / (x_flat.shape[0] * self.top_k)

            # P_i = average router probability per expert
            P_i = router_probs.mean(dim=0)

            # L_aux = alpha * N * sum(f_i * P_i)
            aux_loss = self.num_experts * (f_i * P_i).sum()

        return output.view(batch, seq_len, dim), aux_loss

# ============================================================
# DEMO: Compare dense vs MoE
# ============================================================
dim, hidden = 512, 1024  # Small for demo
num_experts, top_k = 8, 2

dense = SwiGLUExpert(dim, hidden)
moe = MoELayer(dim, hidden, num_experts, top_k)

dense_params = sum(p.numel() for p in dense.parameters())
moe_params = sum(p.numel() for p in moe.parameters())
# Active = shared_expert + top_k * expert_size + router
active_params = dense_params + top_k * dense_params + dim * num_experts

print("=" * 55)
print("DENSE vs MoE COMPARISON")
print("=" * 55)
print(f"Dense FFN params:     {dense_params:,}")
print(f"MoE total params:     {moe_params:,}")
print(f"MoE active per token: ~{active_params:,}")
print(f"Capacity multiplier:  {moe_params / dense_params:.1f}x")
print(f"Compute multiplier:   {active_params / dense_params:.1f}x")

# Test forward pass
x = torch.randn(2, 10, dim)
out, aux = moe(x)
print(f"\nInput:  {x.shape}")
print(f"Output: {out.shape}")
print(f"Aux loss: {aux.item():.4f} (lower = more balanced)")

DENSE vs MoE COMPARISON
Dense FFN params:     1,572,864
MoE total params:     14,159,872
MoE active per token: ~4,722,688
Capacity multiplier:  9.0x
Compute multiplier:   3.0x

Input:  torch.Size([2, 10, 512])
Output: torch.Size([2, 10, 512])
Aux loss: 1.0522 (lower = more balanced)


In [2]:
import torch
import torch.nn.functional as F

# ============================================================
# LOAD BALANCING: THE MAKE-OR-BREAK OF MOE TRAINING
# ============================================================

def compute_aux_loss(router_logits, top_k_indices, num_experts):
    """
    Auxiliary load balancing loss (Switch Transformer style).

    L_aux = alpha * N * sum(f_i * P_i)
 - f_i: fraction of tokens dispatched to expert i
 - P_i: average router probability for expert i
 - Minimized when all experts get equal share
    """
    batch_seq = router_logits.shape[0]

    # f_i: actual token distribution
    f_i = torch.zeros(num_experts)
    for i in range(num_experts):
        f_i[i] = (top_k_indices == i).float().sum()
    f_i = f_i / (batch_seq * top_k_indices.shape[1])

    # P_i: average router probability
    P_i = F.softmax(router_logits, dim=-1).mean(dim=0)

    return num_experts * (f_i * P_i).sum()

# ============================================================
# SCENARIO 1: Balanced routing (healthy MoE)
# ============================================================
print("=" * 60)
print("SCENARIO 1: BALANCED ROUTING (HEALTHY)")
print("=" * 60)

num_experts = 8
num_tokens = 1000

balanced_logits = torch.randn(num_tokens, num_experts)
balanced_probs = F.softmax(balanced_logits, dim=-1)
_, balanced_idx = torch.topk(balanced_probs, 2, dim=-1)

balanced_loss = compute_aux_loss(balanced_logits, balanced_idx, num_experts)

print(f"Aux loss: {balanced_loss.item():.4f}")
print("Expert usage (% of tokens):")
for i in range(num_experts):
    pct = (balanced_idx == i).float().mean().item() * 100
    bar = "#" * int(pct * 2)
    print(f"  Expert {i}: {pct:5.1f}% {bar}")

# ============================================================
# SCENARIO 2: Collapsed routing (broken MoE)
# ============================================================
print(f"\n{'=' * 60}")
print("SCENARIO 2: COLLAPSED ROUTING (BROKEN - NO LOAD BALANCING)")
print("=" * 60)

collapsed_logits = torch.randn(num_tokens, num_experts)
collapsed_logits[:, 0] += 10  # Bias toward expert 0
collapsed_logits[:, 1] += 9   # And expert 1

collapsed_probs = F.softmax(collapsed_logits, dim=-1)
_, collapsed_idx = torch.topk(collapsed_probs, 2, dim=-1)

collapsed_loss = compute_aux_loss(collapsed_logits, collapsed_idx, num_experts)

print(f"Aux loss: {collapsed_loss.item():.4f} (much higher!)")
print("Expert usage (% of tokens):")
for i in range(num_experts):
    pct = (collapsed_idx == i).float().mean().item() * 100
    bar = "#" * int(pct * 2)
    status = " << OVERLOADED" if pct > 30 else (" << WASTED" if pct < 2 else "")
    print(f"  Expert {i}: {pct:5.1f}% {bar}{status}")

# ============================================================
# KEY INSIGHT
# ============================================================
print(f"\n{'=' * 60}")
print("KEY INSIGHT")
print("=" * 60)
print(f"Balanced loss:  {balanced_loss.item():.4f}")
print(f"Collapsed loss: {collapsed_loss.item():.4f}")
print(f"Ratio: {collapsed_loss.item()/balanced_loss.item():.1f}x higher")
print()
print("Without load balancing:")
print(" - 2 experts do ALL the work")
print(" - 6 experts learn NOTHING (wasted parameters)")
print(" - Model is effectively a 2-expert system")
print(" - DeepSeek V3 would waste 250 of 256 experts!")

SCENARIO 1: BALANCED ROUTING (HEALTHY)
Aux loss: 1.0005
Expert usage (% of tokens):
  Expert 0:  12.5% #########################
  Expert 1:  13.2% ##########################
  Expert 2:  12.5% #########################
  Expert 3:  12.7% #########################
  Expert 4:  11.7% #######################
  Expert 5:  13.3% ##########################
  Expert 6:  12.2% ########################
  Expert 7:  11.9% #######################

SCENARIO 2: COLLAPSED ROUTING (BROKEN - NO LOAD BALANCING)
Aux loss: 3.9985 (much higher!)
Expert usage (% of tokens):
  Expert 0:  50.0% #################################################################################################### << OVERLOADED
  Expert 1:  50.0% #################################################################################################### << OVERLOADED
  Expert 2:   0.0%  << WASTED
  Expert 3:   0.0%  << WASTED
  Expert 4:   0.0%  << WASTED
  Expert 5:   0.0%  << WASTED
  Expert 6:   0.0%  << WASTED
  Expert 7:   0.0%  <

In [ ]:
import torch
import torch.nn.functional as F

# ============================================================
# EXPERT PARALLELISM: HOW MOE INFERENCE WORKS AT SCALE
# Simulates distributing 16 experts across 4 "GPUs"
# ============================================================

class SimulatedExpertParallel:
    """
    Simulates expert parallelism across multiple GPUs.

    Real systems use NCCL all-to-all communication.
    This shows the routing + dispatch logic.
    """
    def __init__(self, num_experts=16, num_gpus=4, top_k=2):
        self.num_experts = num_experts
        self.num_gpus = num_gpus
        self.top_k = top_k
        self.experts_per_gpu = num_experts // num_gpus

        # Map expert -> GPU
        self.expert_to_gpu = {}
        for i in range(num_experts):
            self.expert_to_gpu[i] = i // self.experts_per_gpu

    def route_and_dispatch(self, tokens, router_logits):
        """Simulate the full MoE routing pipeline."""
        num_tokens = len(tokens)

        # Step 1: Router selects top-K experts per token
        probs = F.softmax(router_logits, dim=-1)
        top_k_probs, top_k_idx = torch.topk(probs, self.top_k, dim=-1)

        # Step 2: Build dispatch table (which tokens go where)
        gpu_dispatch = {g: [] for g in range(self.num_gpus)}
        for t in range(num_tokens):
            for k in range(self.top_k):
                expert_id = top_k_idx[t, k].item()
                gpu_id = self.expert_to_gpu[expert_id]
                gpu_dispatch[gpu_id].append({
                    'token_idx': t,
                    'token': tokens[t],
                    'expert_id': expert_id,
                    'weight': top_k_probs[t, k].item()
                })

        return gpu_dispatch, top_k_idx, top_k_probs

    def print_dispatch(self, tokens, router_logits):
        dispatch, idx, probs = self.route_and_dispatch(tokens, router_logits)

        print("=" * 60)
        print("EXPERT PARALLELISM: TOKEN DISPATCH ACROSS GPUS")
        print("=" * 60)

        # Show routing decisions
        print("\nROUTING DECISIONS:")
        for t, token in enumerate(tokens):
            experts = [idx[t, k].item() for k in range(self.top_k)]
            weights = [f"{probs[t, k].item():.2f}" for k in range(self.top_k)]
            gpus = [self.expert_to_gpu[e] for e in experts]
            print(f"  '{token}' -> Experts {experts} "
                  f"(weights: {weights}) -> GPUs {gpus}")

        # Show GPU workloads
        print(f"\nGPU WORKLOADS:")
        for gpu_id in range(self.num_gpus):
            expert_range = f"{gpu_id * self.experts_per_gpu}-"
            expert_range += f"{(gpu_id + 1) * self.experts_per_gpu - 1}"
            num_tasks = len(dispatch[gpu_id])
            bar = "#" * num_tasks
            print(f"  GPU {gpu_id} (experts {expert_range}): "
                  f"{num_tasks} tasks {bar}")

        # Communication cost
        cross_gpu = 0
        for t in range(len(tokens)):
            gpus = set()
            for k in range(self.top_k):
                gpus.add(self.expert_to_gpu[idx[t, k].item()])
            if len(gpus) > 1:
                cross_gpu += 1

        print(f"\nCROSS-GPU COMMUNICATION:")
        print(f"  {cross_gpu}/{len(tokens)} tokens need inter-GPU transfer")
        print(f"  ({cross_gpu/len(tokens)*100:.0f}% of tokens cross GPU boundaries)")

        return dispatch

# ============================================================
# DEMO: Route code tokens through 16 experts on 4 GPUs
# ============================================================

tokens = ["def", "fibonacci", "(", "n", ")", ":",
          "if", "n", "<=", "1"]

# Simulated router scores (biased toward relevant experts)
num_experts = 16
logits = torch.randn(len(tokens), num_experts) * 0.5
# Bias code tokens toward code experts (3, 7)
for i in range(len(tokens)):
    logits[i, 3] += 2.0   # Code expert
    logits[i, 7] += 1.5   # Python expert
# Bias syntax tokens differently
logits[2, 1] += 3.0  # "(" -> syntax expert
logits[4, 1] += 3.0  # ")" -> syntax expert
logits[5, 8] += 3.0  # ":" -> punctuation expert

ep = SimulatedExpertParallel(num_experts=16, num_gpus=4, top_k=2)
ep.print_dispatch(tokens, logits)

print(f"\n{'=' * 60}")
print("DEEPSEEK V3 AT SCALE")
print("=" * 60)
print("""
Real DeepSeek V3 inference:
 - 256 experts across 8-16 H100 GPUs
 - Each GPU hosts 16-32 experts
 - All-to-all NCCL communication for token dispatch
 - ~5-10% overhead from routing + communication
 - Net result: GPT-4 quality at ~1/3 the inference cost
""")